# Risk Management Silver Layer

---

## Imports


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

## Read Data

In [0]:
def read_data(table_name):
    """
    Read bronze table.

    Args:
        table_name(str): Name of the table to read

    Returns:
        DataFrame: Spark DataFrame
    """

    df = spark.table(table_name)

    return df

## Calculate Daily Return

In [0]:
def calculate_returns(df):
    """
    Calculate daily returns.

    Formula:
        (close / previous_close) - 1

    Args:
        df: Spark DataFrame

    Returns:
        Spark DataFrame with daily returns
    """

    window_spec = Window \
        .partitionBy("ticker") \
        .orderBy("date")

    df = df \
        .withColumn("previous_close",F.lag("close").over(window_spec)) \
        .withColumn("return_pct",(F.col("close") / F.col("previous_close")) - 1)

    return df

## Calculate Rolling Metrics

In [0]:
def calculate_rolling_metrics(df):
    """
    Calculate rolling risk metrics.

    Args:
        df: Spark DataFrame

    Returns:
        Spark DataFrame
    """

    window_30d = Window \
        .partitionBy("ticker") \
        .orderBy("date") \
        .rowsBetween(-29, 0)

    df = df \
        .withColumn("volatility_30d", F.stddev("return_pct").over(window_30d)) \
        .withColumn("avg_volume_30d", F.avg("volume").over(window_30d)) \
        .withColumn("return_mean_30d", F.avg("return_pct").over(window_30d)) \
        .withColumn("volume_std_30d", F.stddev("volume").over(window_30d)) \
        .withColumn("max_return_30d", F.max("return_pct").over(window_30d)) \
        .withColumn("min_return_30d", F.min("return_pct").over(window_30d)) \
        .withColumn("var_95", F.abs(F.col("return_mean_30d") - (1.65 * F.col("volatility_30d"))))

    return df

## Clean Dataset

In [0]:
def clean_dataset(df):
    """
    Remove helper columns and null records.

    Args:
        df: Spark DataFrame

    Returns:
        Spark DataFrame
    """

    df = df \
        .drop("previous_close") \
        .filter(F.col("return_pct").isNotNull())

    return df

## Save Data

In [0]:
def save_data(df, table_name):
    """
    Save data to a Delta table.

    Args:
        df (DataFrame): Spark DataFrame
        table_name (str): Name of the Delta table

    Returns:
        None
    """

    df.write.format("delta") \
        .mode('overwrite') \
        .option("overwriteSchema", "true") \
        .option("mergeSchema", "true") \
        .saveAsTable(table_name)

## Main Function

In [0]:
def main():
    """
    Execute silver layer.
    """

    df = read_data("risk_management.risk_dataset_bronze")
    df = calculate_returns(df)
    df = calculate_rolling_metrics(df)
    df = clean_dataset(df)
    save_data(df, "risk_management.risk_dataset_silver")

## Execution

In [0]:
if __name__ == "__main__":
    main()